# Buscando Diferencias entre Grupos Pareados

## ANOVA de medidas repetidas con 2 factores

### Procedimiento
- Importar librerias
- Cargar la primera hoja de trabajo de excel en Pandas
- Limpiar los datos
- Prueba de Normalidad (SHAPIRO-WILK)
- Prueba de esfericidad (W de MAUCHLY)
- Prueba ANOVA de medidas repetidas de 2 factores
- Para la categoria con 3 o mas grupos HSD Tukey

In [ ]:
#__ Cargar librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import shapiro
from scipy.stats import tukey_hsd
from pingouin import sphericity
from pingouin import rm_anova

In [ ]:
#___ Cargar la hoja de trabajo en un dataframe
anovamr2_df = pd.read_excel('datos_analisis_estadistico.xlsx', sheet_name = 'ANOVA 2 MED REP')
anovamr2_df

In [ ]:
#___ PARTE 1
##___ Transformar la forma del dataframe para utilizar el test de normalidad (SHAPIRO-WILK)
###___ Los datos deben estar en formato largo para la prueba anova de medidas repetidas de dos factores
melted_anovamr2_df = anovamr2_df.melt(id_vars='ALUMNO', value_vars=['TeorT1', 'TeorT2', 'TeorT3', 'PracT1','PracT2','PracT3'], var_name='TIEMPO', value_name='CALIFICACION')
melted_anovamr2_df

In [ ]:
#___ PARTE 2
melted_anovamr2_df['EXAMEN'] = melted_anovamr2_df['TIEMPO'].apply(lambda x: 'T' if 'Teor' in x else 'P')
melted_anovamr2_df['PARCIAL'] = melted_anovamr2_df['TIEMPO'].apply(lambda x: '1ro' if 'T1' in x else ('2do' if 'T2' in x else '3ro'))
melted_anovamr2_df.drop('TIEMPO', axis=1)

In [ ]:
#___ Prueba de normalidad METODO A (RESIDUOS)
modelo = smf.mixedlm("CALIFICACION ~ C(PARCIAL) * C(EXAMEN)", melted_anovamr2_df, groups=melted_anovamr2_df["ALUMNO"])
modelo_ajustado = modelo.fit()

##___ Se extraen los residuos
residuos = modelo_ajustado.resid

In [ ]:
#___ Prueba de normalidad METODO A (VISUAL)
sm.qqplot(residuos, line='s')
plt.title("Grafica Q-Q del Modelo de Residuos")
plt.show()

In [ ]:
#___ Prueba de normalidad METODO B (ANALITICA SHAPIRO-WILK)
###___ La funcion sp.shapiro() verifica que una o mas columnas en el dataframe siguen una distribucion normal.
prueba_normalidad_B = shapiro(residuos)
prueba_normalidad_B

In [ ]:
#___ Prueba de esfericidad (W de MAUCHLY) FORMATO LARGO
prueba_esfe = sphericity(data=melted_anovamr2_df, dv='CALIFICACION', within=['PARCIAL', 'EXAMEN'], subject='ALUMNO')
prueba_esfe
#%%% FALLO PRUEBA DE ESFERICIDAD:
#%%% ES NECESARIO APLICAR LA CORRECCION GREENHOUSE-GEISSER EN LA PRUEBA ANOVA

In [ ]:
#___ Prueba de ANOVA de medidas repetidas de 2 factores
prueba_anovamr2 = rm_anova(data=melted_anovamr2_df, dv='CALIFICACION', within=['PARCIAL', 'EXAMEN'], subject='ALUMNO', correction=True, detailed=True)
prueba_anovamr2

In [ ]:
#___ Prueba Tukey 'PARCIAL' (Libreria SciPy)
prueba_tukey_A = tukey_hsd(melted_anovamr2_df[melted_anovamr2_df['PARCIAL'] == '1ro']['CALIFICACION'], melted_anovamr2_df[melted_anovamr2_df['PARCIAL'] == '2do']['CALIFICACION'], melted_anovamr2_df[melted_anovamr2_df['PARCIAL'] == '3ro']['CALIFICACION'])
print(prueba_tukey_A)


In [ ]:
#___ Grafica de error promedio
##___ Prepararr el dataframe
promedio_t1 = anovamr2_df['TeorT1'].mean()
promedio_t2 = anovamr2_df['TeorT2'].mean()
promedio_t3 = anovamr2_df['TeorT3'].mean()

promedio_p1 = anovamr2_df['PracT1'].mean()
promedio_p2 = anovamr2_df['PracT2'].mean()
promedio_p3 = anovamr2_df['PracT3'].mean()

destd_t1 = anovamr2_df['TeorT1'].std()
destd_t2 = anovamr2_df['TeorT2'].std()
destd_t3 = anovamr2_df['TeorT3'].std()

destd_p1 = anovamr2_df['PracT1'].std()
destd_p2 = anovamr2_df['PracT2'].std()
destd_p3 = anovamr2_df['PracT3'].std()

plt.bar(['INICIO','MEDIO','FINAL'], [promedio_t1, promedio_t2, promedio_t3], yerr=[destd_t1, destd_t2, destd_t3], width=-0.3, align='edge', capsize=10, label='TEORICO')
plt.bar(['INICIO','MEDIO','FINAL'], [promedio_p1, promedio_p2, promedio_p3], yerr=[destd_p1, destd_p2, destd_p3], width=0.3, align='edge', capsize=10, label='PRACTICO')
plt.legend()
plt.xlabel('TIEMPO DE APLICACION DEL EXAMEN')
plt.ylabel('CALIFICACION')
plt.title('CALIFICACIONES DURANTE EL SEMESTRE')
plt.ylim(0,10)